В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [43]:
import pandas as pd
import numpy as np
import process_bank_churn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
import time

In [2]:
df = pd.read_csv('train.csv')

In [3]:
X_train, y_train, X_val, y_val, input_cols, scaler, encoder = process_bank_churn.preprocess_data(df)

1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [4]:
n_neighbors = 5

In [5]:
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

KNeighborsClassifier()

In [6]:
train_prob = knn.predict_proba(X_train)[:, 1]
val_prob = knn.predict_proba(X_val)[:, 1]

In [7]:
train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)
print('Train AUROC:', train_auc)
print('Validation AUROC:', val_auc)

Train AUROC: 0.959994556275159
Validation AUROC: 0.8641594073667604


Висновок: результат показує, що модель добре навчилась на тренувальних даних і гірше працює з валідаційним набором даних. У даному випадку -- high variance, модель перенавчена.

2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [8]:
grid_param = {'n_neighbors': range(1, 31)}

In [12]:
grid = GridSearchCV(
    estimator = KNeighborsClassifier(),
    param_grid = grid_param,
    cv = 5,
    scoring = 'roc_auc',
    verbose = 1
)

In [14]:
grid.fit(X_train, y_train)
print('The best n_eighbors:', grid.best_params_['n_neighbors'])

Fitting 5 folds for each of 30 candidates, totalling 150 fits
The best n_eighbors: 30


In [15]:
knn_best = grid.best_estimator_

In [16]:
train_prob = knn_best.predict_proba(X_train)[:, 1]
val_prob = knn_best.predict_proba(X_val)[:, 1]
train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)
print('Train AUROC:', train_auc)
print('Validation AUROC:', val_auc)

Train AUROC: 0.9302734724109489
Validation AUROC: 0.9126637629467043


Висновок: модель стала значно краще, ніж була у поперньому пункті. Різниці майже немає між тренованим і валідаційним наборами, що говорить про хорошу генералізацію. Щодо дерев прийняття рішень з попереднього д/з, я мала такий самий результат 0.91

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [23]:
param_grid = {'max_depth': range(1, 21, 1),
              'max_leaf_nodes': range(2, 11),
              }

In [24]:
dt = DecisionTreeClassifier(random_state=42)

In [25]:
grid_dt = GridSearchCV(
    dt,
    param_grid = param_grid,
    cv = 3,
    scoring = 'roc_auc',
    verbose = 1
)

In [26]:
# time
start_time = time.time()
grid_dt.fit(X_train, y_train)
end_time = time.time()
search_time = end_time - start_time
print('Search time - seconds:', search_time)

Fitting 3 folds for each of 180 candidates, totalling 540 fits
Search time - seconds: 16.16233992576599


/Users/macbook/anaconda3/lib/python3.12/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [27]:
print(grid_dt.best_params_)

{'max_depth': 5, 'max_leaf_nodes': 10}


In [28]:
dt_best = grid_dt.best_estimator_

In [29]:
train_prob = dt_best.predict_proba(X_train)[:, 1]
val_prob = dt_best.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)

print('Train AUROC:', train_auc)
print('Validation AUROC:', val_auc)

Train AUROC: 0.9014754782174744
Validation AUROC: 0.9002184649152891


Висновок: тренувальний і валідаційний набори показали однакові результати 0.90 -- модель Дерев прийняття рішень хороша: добре узагальнює, не перенавчена. Однак, трохи поступається попередній моделі kNN 0.91.

4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [37]:
params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

In [38]:
dt = DecisionTreeClassifier(random_state=42)

In [39]:
random_search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=params_dt,
    n_iter=40,
    cv = 3,
    scoring='roc_auc',
    verbose=1,
    n_jobs=1
)

In [40]:
start_time = time.time()
random_search.fit(X_train, y_train)
end_time = time.time()
search_time = end_time - start_time
print('Search time - seconds:', search_time)
print('The best params:', random_search.best_params_)

Fitting 3 folds for each of 40 candidates, totalling 120 fits
Search time - seconds: 2.5087637901306152
The best params: {'splitter': 'best', 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_leaf_nodes': 14, 'max_features': None, 'max_depth': 9, 'criterion': 'gini'}


In [41]:
dt_random_search_best = random_search.best_estimator_

In [42]:
train_prob = dt_random_search_best.predict_proba(X_train)[:, 1]
val_prob = dt_random_search_best.predict_proba(X_val)[:, 1]
train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)
print('Train AUROC:', train_auc)
print('Validation AUROC:', val_auc)

Train AUROC: 0.9130523692670586
Validation AUROC: 0.9127639069895055


Висновок: у результаті отримано схожий результат в RandomizedSearchCV 0.91 та GridSearchCV 0.90, остання модель трохи краще. Цей спосіб є дещо продуктивнішим, ажде має ширші параметри пошуку.

5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉

Сабміт не роблю, минулого разу теж був результат 0.91